In [33]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

print(np.__version__)
print(torch.__version__)

1.26.4
2.2.2


In [2]:
import pandas as pd

In [3]:
raw_train = pd.read_csv(
    "../data/01_raw/train.csv", parse_dates=["DateTimeOfAccident", "DateReported"]
)
raw_test = pd.read_csv(
    "../data/01_raw/test.csv", parse_dates=["DateTimeOfAccident", "DateReported"]
)

In [4]:
df = raw_train.copy()

In [5]:
df_desc = df["ClaimDescription"]

In [6]:
# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

In [19]:
desc_list = df["ClaimDescription"].to_list()

In [25]:
desc_embeddings_test = model.encode(desc_list, batch_size=32, show_progress_bar=True)

Batches: 100%|██████████| 1688/1688 [45:21<00:00,  1.61s/it]


In [27]:
desc_embeddings_test.shape

(54000, 1024)

In [31]:
pd.DataFrame(desc_embeddings_test).to_csv("../data/02_intermediate/desc_embeddings.csv")

In [22]:
desc_embeddings_test

array([ 0.00148733,  0.0299702 , -0.01144153, ..., -0.01122504,
        0.01392182,  0.01257017], dtype=float32)

In [38]:
pd.DataFrame(desc_embeddings_test)

,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,0.001487,0.029970,-0.011442,0.048489,-0.002995,0.026020,-0.042131,-0.001255,0.002140,0.089078,...,0.010866,-0.020232,0.011439,-0.013687,0.038593,0.013632,-0.018313,-0.011225,0.013922,0.012570
1,0.018784,0.041519,-0.012378,0.006399,-0.025692,0.002942,-0.004686,0.012054,-0.016261,0.085406,...,0.021550,-0.005203,-0.004713,-0.017795,0.033185,0.022527,0.002149,0.009540,0.000278,0.012832
2,-0.014101,-0.050322,-0.008808,-0.056488,0.000461,0.012127,0.004506,0.014728,0.003286,0.083662,...,0.024515,-0.033829,0.037780,-0.000514,0.006697,-0.000888,0.020751,-0.001497,0.000164,-0.008888
3,-0.005803,-0.014721,-0.012542,0.001302,0.043809,0.036360,-0.061608,-0.031926,-0.004644,0.056719,...,0.000502,0.030812,-0.021745,0.010949,0.051259,0.001258,0.020859,0.015336,-0.011571,0.024081
4,-0.009875,0.012587,-0.011190,0.006387,0.040640,0.049939,-0.069833,0.014698,0.005110,0.084298,...,0.019505,0.013858,-0.010850,-0.011647,0.012808,-0.037370,-0.023120,-0.026595,0.013502,0.015772
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53995,-0.003562,0.010075,-0.008890,0.001484,0.009993,0.008711,-0.065561,0.002855,0.052810,0.064419,...,-0.003305,0.026201,0.012989,-0.009885,0.060643,-0.007575,0.011340,-0.012153,0.038601,-0.008200
53996,0.015112,0.012150,-0.010776,-0.016826,0.028332,0.023691,-0.072692,-0.009050,0.024260,0.043973,...,0.004346,0.033768,0.008969,-0.024820,0.059359,0.038037,0.010838,-0.001083,0.016130,-0.023909
53997,-0.002457,0.015267,-0.009333,-0.017677,0.009632,0.003854,-0.008203,-0.034006,0.009554,0.090103,...,-0.019523,-0.015677,0.004463,0.003269,0.027762,0.025800,0.008397,0.011369,0.003527,0.000265
53998,0.024932,-0.010606,-0.013877,-0.031215,0.043794,0.009136,-0.004920,-0.003005,0.062175,0.060934,...,0.011932,-0.002758,-0.002658,-0.022954,0.065037,-0.000319,0.013647,-0.051996,0.007645,-0.042052


In [43]:
import sqlite3

df_embeddings = df[["ClaimNumber"]].join(pd.DataFrame(desc_embeddings_test))

# Save dataframe as a SQL table
conn = sqlite3.connect("../data/02_intermediate/embeddings/desc_embeddings.db")

df_embeddings.to_sql(name="desc_embeddings", con=conn, if_exists="fail", index=False)

conn.close()

In [4]:
import sqlite3
import pandas as pd

# Load desc embeddings SQL table
conn = sqlite3.connect("../data/02_intermediate/embeddings/desc_embeddings.db")

df_emb_loaded = pd.read_sql("SELECT * FROM desc_embeddings", conn)

conn.close()

In [5]:
df_emb_loaded

,ClaimNumber,0,1,2,3,4,5,6,7,8,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,WC8285054,0.001487,0.029970,-0.011442,0.048489,-0.002995,0.026020,-0.042131,-0.001255,0.002140,...,0.010866,-0.020232,0.011439,-0.013687,0.038593,0.013632,-0.018313,-0.011225,0.013922,0.012570
1,WC6982224,0.018784,0.041519,-0.012378,0.006399,-0.025692,0.002942,-0.004686,0.012054,-0.016261,...,0.021550,-0.005203,-0.004713,-0.017795,0.033185,0.022527,0.002149,0.009540,0.000278,0.012832
2,WC5481426,-0.014101,-0.050322,-0.008808,-0.056488,0.000461,0.012127,0.004506,0.014728,0.003286,...,0.024515,-0.033829,0.037780,-0.000514,0.006697,-0.000888,0.020751,-0.001497,0.000164,-0.008888
3,WC9775968,-0.005803,-0.014721,-0.012542,0.001302,0.043809,0.036360,-0.061608,-0.031926,-0.004644,...,0.000502,0.030812,-0.021745,0.010949,0.051259,0.001258,0.020859,0.015336,-0.011571,0.024081
4,WC2634037,-0.009875,0.012587,-0.011190,0.006387,0.040640,0.049939,-0.069833,0.014698,0.005110,...,0.019505,0.013858,-0.010850,-0.011647,0.012808,-0.037370,-0.023120,-0.026595,0.013502,0.015772
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53995,WC9370727,-0.003562,0.010075,-0.008890,0.001484,0.009993,0.008711,-0.065561,0.002855,0.052810,...,-0.003305,0.026201,0.012989,-0.009885,0.060643,-0.007575,0.011340,-0.012153,0.038601,-0.008200
53996,WC8396269,0.015112,0.012150,-0.010776,-0.016826,0.028332,0.023691,-0.072692,-0.009050,0.024260,...,0.004346,0.033768,0.008969,-0.024820,0.059359,0.038037,0.010838,-0.001083,0.016130,-0.023909
53997,WC3609528,-0.002457,0.015267,-0.009333,-0.017677,0.009632,0.003854,-0.008203,-0.034006,0.009554,...,-0.019523,-0.015677,0.004463,0.003269,0.027762,0.025800,0.008397,0.011369,0.003527,0.000265
53998,WC5038565,0.024932,-0.010606,-0.013877,-0.031215,0.043794,0.009136,-0.004920,-0.003005,0.062175,...,0.011932,-0.002758,-0.002658,-0.022954,0.065037,-0.000319,0.013647,-0.051996,0.007645,-0.042052
